# EXPERIMENT 3

# AIM

To understand and implement proper data preprocessing using Scikit-learn pipelines, identify data leakage in a machine learning workflow, and compare the performance of a leaky preprocessing pipeline with a correctly implemented leakage-free pipeline.

# OBJECTIVES

1. To understand the concept of **data leakage** in machine learning.
2. To identify leakage caused by preprocessing the complete dataset before train-test splitting.
3. To correctly divide the dataset into training and testing sets.
4. To apply **feature scaling** using `StandardScaler`.
5. To encode categorical variables using `OneHotEncoder`.
6. To use `ColumnTransformer` for applying different preprocessing techniques to different feature types.
7. To construct an end-to-end machine learning **Pipeline**.
8. To ensure that preprocessing is fitted only on the training data.
9. To compare the performance of leaky and corrected pipelines.
10. To understand how leakage can produce **optimistic and unreliable test performance**.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [4]:
df = pd.read_csv("Lab_3_dataset.csv")

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (152, 16)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,income_marker
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             152 non-null    int64
 1   workclass       152 non-null    str  
 2   fnlwgt          152 non-null    int64
 3   education       152 non-null    str  
 4   education-num   152 non-null    int64
 5   marital-status  152 non-null    str  
 6   occupation      152 non-null    str  
 7   relationship    152 non-null    str  
 8   race            152 non-null    str  
 9   gender          152 non-null    str  
 10  capital-gain    152 non-null    int64
 11  capital-loss    152 non-null    int64
 12  hours-per-week  152 non-null    int64
 13  native-country  152 non-null    str  
 14  income          152 non-null    str  
 15  income_marker   152 non-null    int64
dtypes: int64(7), str(9)
memory usage: 19.1 KB


In [6]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
gender            0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
income_marker     0
dtype: int64


In [10]:
X = df.drop("income", axis=1)
y = df["income"]

print("Features:")
display(X.head())

print("Target:")
display(y.head())

Features:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income_marker
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


Target:


0    <=50K
1    <=50K
2    <=50K
3    <=50K
4    <=50K
Name: income, dtype: str

In [12]:
X = X.drop(["marital-status", "gender", "race", "relationship"], axis=1)

print("Updated Features:")
display(X.head())

Updated Features:


,age,workclass,fnlwgt,education,education-num,occupation,capital-gain,capital-loss,hours-per-week,native-country,income_marker
0,39,State-gov,77516,Bachelors,13,Adm-clerical,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Exec-managerial,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Handlers-cleaners,0,0,40,United-States,0
3,53,Private,234721,11th,7,Handlers-cleaners,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Prof-specialty,0,0,40,Cuba,0


In [13]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'income_marker']
Categorical Features: ['workclass', 'education', 'occupation', 'native-country']


/var/folders/4d/bxl021h50qdcr6z35z1s85hm0000gn/T/ipykernel_52878/1784015484.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set:", X_test.shape)

Training Set: (106, 11)
Testing Set: (46, 11)


In [15]:
X_leaky = X.copy()

# Leakage occurs here because the scaler sees the complete dataset
scaler = StandardScaler()

X_leaky[numerical_features] = scaler.fit_transform(
    X_leaky[numerical_features]
)

# Encode categorical variables before splitting
X_leaky = pd.get_dummies(
    X_leaky,
    columns=categorical_features,
    drop_first=True
)

# Handle any remaining missing values
X_leaky = X_leaky.fillna(X_leaky.median(numeric_only=True))
X_leaky = X_leaky.fillna(0)

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

model_leaky = LogisticRegression(max_iter=1000)

model_leaky.fit(X_train_leaky, y_train_leaky)

y_pred_leaky = model_leaky.predict(X_test_leaky)

leaky_accuracy = accuracy_score(y_test_leaky, y_pred_leaky)

print("Leaky Pipeline Accuracy:", leaky_accuracy)

Leaky Pipeline Accuracy: 1.0


In [16]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [17]:
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

correct_accuracy = accuracy_score(y_test, y_pred)

print("Corrected Pipeline Accuracy:", correct_accuracy)

Corrected Pipeline Accuracy: 1.0


In [18]:
comparison = pd.DataFrame({
    "Pipeline": ["Leaky Pipeline", "Corrected Pipeline"],
    "Test Accuracy": [leaky_accuracy, correct_accuracy]
})

display(comparison)

,Pipeline,Test Accuracy
0,Leaky Pipeline,1.0
1,Corrected Pipeline,1.0


In [19]:
difference = leaky_accuracy - correct_accuracy

print("Difference in Accuracy:", round(difference, 4))

Difference in Accuracy: 0.0


# OBSERVATION

The leaky preprocessing workflow was successfully identified and compared with a corrected leakage-free pipeline.

In the leaky workflow, `StandardScaler` was fitted on the complete dataset before the train-test split. Therefore, information from the test features was indirectly used during preprocessing.

In the corrected workflow, the dataset was first divided into training and testing sets. The preprocessing steps were then fitted only on the training data through the Scikit-learn `Pipeline`.

The obtained test accuracies of the two approaches were compared. The leaky pipeline may show slightly different or optimistic performance because information from the test set was used during preprocessing.

The corrected pipeline provides a more reliable estimate of how the model will perform on unseen data.

# CONCLUSION

Data preprocessing and model training were successfully implemented using Scikit-learn. The experiment demonstrated that preprocessing the complete dataset before splitting can introduce **data leakage** and result in an unreliable evaluation.

A leakage-free pipeline was constructed using `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `ColumnTransformer`, and `Pipeline`. By fitting preprocessing only on the training data, the test set remained independent of the training process.

Thus, **the correct order is to split the data first and perform preprocessing through a pipeline fitted only on the training data**.